# 05. Exploratory Data Analysis (EDA)

This notebook connects to the SQLite database, uses the summary views defined in `sql/analysis_queries.sql`,
and explores relationships between economic indicators, education investment, enrollment/completion,
and education quality (PISA).

## Setup: Imports and database connection

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..")
DB_PATH = PROJECT_ROOT / "world_indicators.db"

conn = sqlite3.connect(DB_PATH)
print("Connected to:", DB_PATH)

def run_sql(query: str, params=None) -> pd.DataFrame:
    """Helper to run a SQL query and return a DataFrame."""
    return pd.read_sql(query, conn, params=params)

## Ensure analysis views exist

The views `v_country_summary`, `v_subregion_summary`, and `v_continent_summary` are defined in
`sql/analysis_queries.sql`. Run the cell below once to (re)create them inside the database.

In [ ]:
analysis_sql_path = PROJECT_ROOT / "sql" / "analysis_queries.sql"
with open(analysis_sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()
conn.executescript(sql_script)
print("Views created from:", analysis_sql_path)

## 1. Country-level summary

Load the `v_country_summary` view into a DataFrame for analysis.

In [ ]:
country = run_sql("SELECT * FROM v_country_summary WHERE avg_pisa > 0")
print(country.shape)
country.head()

### Basic stats and correlations

Look at distributions and correlations between key numeric variables.

In [ ]:
num_cols = [
    "avg_gdp_per_capita_2020_usd",
    "avg_gov_exp_edu_gdp_pct_20y",
    "avg_primary_enrollment_pct",
    "avg_primary_completion_pct",
    "avg_secondary_enrollment_pct",
    "avg_secondary_completion_pct",
    "avg_tertiary_enrollment_pct",
    "avg_tertiary_completion_pct",
    "avg_pisa_reading",
    "avg_pisa_mathematics",
    "avg_pisa_science",
    "avg_pisa",
]

summary = country[num_cols].describe().T
summary

In [ ]:
corr = country[num_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=False)
plt.title("Correlation matrix of key indicators")
plt.tight_layout()

## 2. Relationship between GDP per capita and PISA

Scatter plot of GDP per capita vs. average PISA score, colored by continent.

In [ ]:
country_pisa = country.dropna(subset=["avg_gdp_per_capita_2020_usd", "avg_pisa"]).copy()
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=country_pisa,
    x="avg_gdp_per_capita_2020_usd",
    y="avg_pisa",
    hue="continent",
    alpha=0.8,
)
plt.xscale("log")  # GDP is highly skewed; log scale helps
plt.xlabel("GDP per capita 2020 (USD, log scale)")
plt.ylabel("Average PISA score")
plt.title("GDP per capita vs. PISA performance by country")
plt.tight_layout()

## 3. Education spending vs PISA

Does higher government expenditure on education (as % of GDP) relate to better PISA outcomes?

In [ ]:
spend_pisa = country.dropna(subset=["avg_gov_exp_edu_gdp_pct_20y", "avg_pisa"]).copy()
plt.figure(figsize=(9, 6))
sns.regplot(
    data=spend_pisa,
    x="avg_gov_exp_edu_gdp_pct_20y",
    y="avg_pisa",
    scatter_kws={"alpha": 0.6},
    line_kws={"color": "black"},
)
plt.xlabel("Avg gov expenditure on education (% of GDP, last 20 years)")
plt.ylabel("Average PISA score")
plt.title("Education spending vs. PISA performance (all countries)")
plt.tight_layout()

## 4. Enrollment vs completion by continent

Use the continent-level view to see how enrollment and completion vary across regions
for different education levels.

In [ ]:
continent = run_sql("SELECT * FROM v_continent_summary")
continent

In [ ]:
melt_cols = [
    "avg_primary_enrollment_pct",
    "avg_primary_completion_pct",
    "avg_secondary_enrollment_pct",
    "avg_secondary_completion_pct",
    "avg_tertiary_enrollment_pct",
    "avg_tertiary_completion_pct",
]
continent_long = continent.melt(
    id_vars=["continent"], value_vars=melt_cols,
    var_name="metric", value_name="value"
)
plt.figure(figsize=(10, 6))
sns.barplot(
    data=continent_long,
    x="continent",
    y="value",
    hue="metric"
)
plt.ylabel("Percentage")
plt.title("Enrollment and completion by continent and education level")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

## 5. Extended angle: within-continent inequality in PISA

For each continent, compare distribution of PISA scores (min, median, max) to highlight
+    "inequality" in outcomes between countries within the same region.

In [ ]:
pisa_by_continent = (
    country.dropna(subset=["avg_pisa"])
    .groupby("continent")["avg_pisa"]
    .agg(["min", "median", "max", "count"])
    .reset_index()
)
pisa_by_continent

In [ ]:
plt.figure(figsize=(8, 5))
for cont, grp in country.dropna(subset=["avg_pisa"]).groupby("continent"):
    sns.kdeplot(grp["avg_pisa"], label=cont, fill=False, linewidth=1.5)
plt.xlabel("Average PISA score")
plt.title("Distribution of PISA scores by continent")
plt.legend()
plt.tight_layout()